
# Lab 4 · Adding Trajectory Evaluations — opencode go 版（收敛分数）

这是教学视频 **Lab 4: Adding Trajectory Evaluations** 的本地复刻版，沿用 `agent1_phoneix_lab.ipynb` 搭好的本地设施（opencode go 订阅的 `mimo-v2.5`、本地 Phoenix）。

教程（`L9.ipynb`）基于旧版 `phoenix.experiments` API；本环境是 **arize-phoenix 20.3.0 + 内置 phoenix-client 3.2.0**，实验入口全部迁到了 `phoenix.client.Client().experiments` 下。对应关系：

| 教程写法（旧） | 本 Notebook 写法（phoenix-client 3.2.0） |
|---|---|
| `px = phoenix as px; px_client = px.Client()` | `from phoenix.client import Client; px_client = Client(base_url=...)` |
| `px_client.upload_dataset(dataframe=..., dataset_name=..., input_keys=...)` | `px_client.datasets.create_dataset(name=..., dataframe=..., input_keys=...)` |
| `from phoenix.experiments import run_experiment, evaluate_experiment` | `px_client.experiments.run_experiment / evaluate_experiment` |
| `from phoenix.experiments.types import Example` | 不需要 import；task 单参数命名 `example` 即绑定完整样例对象，兼容旧式 `example.input[...]` 访问 |
| `@create_evaluator(name=..., kind="CODE")` | 直接传普通函数；返回 `float` 或 `(score, explanation)` 元组或 `{"score":..., "label":..., "explanation":...}` dict |
| `experiment.as_dataframe()` | 从返回的 `experiment["task_runs"]` 手动组装 DataFrame（见下文代码） |

**这一站解决「走得稳不稳」**：结果级评估只看最终答案对不对，轨迹级评估看 Agent 走了多长的路。**收敛分数（Convergence Score）** 度量的是——同一个问题的不同问法，Agent 执行路径的长度是否一致。路径忽长忽短，说明路由/规划不稳定。

## PPT 四步法

$$\text{Overall Convergence Score} = \frac{\sum_{i=1}^{N} \min\left(1,\ \frac{S_{\text{optimal}}}{S_{\text{agent},i}}\right)}{N}$$

1. 在一组**语义等价的问法**上跑 Agent（N = 问法数）；
2. 记录每次运行的路径步数 $S_{\text{agent},i}$；
3. 取全批最短路径作最优基准 $S_{\text{optimal}}$；
4. 逐条算 $\min(1,\ S_{\text{optimal}}/S_{\text{agent},i})$，取均值。

**前置条件：**

- 本地 Phoenix 已启动：`tests/scripts/start-phoenix-local.sh`，浏览器打开 <http://127.0.0.1:6006>
- Kernel 选择 `Python (AI Interviewer · Phoenix Lab)`（即 `ai_interviewer/.venv`，依赖已齐）
- 模型上游可用性会波动，建议先跑「探活」cell 确认 `mimo-v2.5` 在线
- 本 Notebook 会真实调用模型跑 17 个问法（每个 1~2 次 LLM 调用），预计 2~4 分钟


In [ ]:

import json
import os
import subprocess
from datetime import datetime
from pathlib import Path

from openai import OpenAI

# Phoenix OTel tracing
from phoenix.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor

# Phoenix 实验客户端
from phoenix.client import Client
import pandas as pd



## 初始化 opencode go 客户端

与 `agent1_phoneix_lab.ipynb` 完全一致：`base_url` 指向 opencode go 官方端点，API key 优先读环境变量、否则从 macOS Keychain 读取。

```text
探活：curl POST https://opencode.ai/zen/go/v1/chat/completions（模型 mimo-v2.5，Chat Completions 协议）
```


In [ ]:

def get_opencode_go_api_key() -> str:
    """优先读环境变量 OPENCODE_GO_API_KEY；否则从 macOS Keychain 动态读取。"""
    key = os.environ.get("OPENCODE_GO_API_KEY")
    if key:
        return key
    user = os.environ.get("USER", "")
    result = subprocess.run(
        ["security", "find-generic-password", "-a", user, "-s", "opencode-go-api-key", "-w"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0 or not result.stdout.strip():
        raise RuntimeError(
            "未找到 opencode go API key：请 export OPENCODE_GO_API_KEY=<your key>，"
            "或确认 Keychain 中存在服务名 opencode-go-api-key"
        )
    return result.stdout.strip()


client = OpenAI(
    api_key=get_opencode_go_api_key(),
    base_url="https://opencode.ai/zen/go/v1",  # opencode go 官方端点（Chat Completions 协议）
)

MODEL = "mimo-v2.5"  # 官方端点表：/chat/completions；价格低（$0.14/$0.22 每 1M tokens）、月额度大


# ── 探活：确认模型在线再跑实验 ─────────────────────────────────────────
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "ping"}],
    max_tokens=5,
)
print("模型在线:", (resp.choices[0].message.content or "(空回复)")[:20])



## 接入 Phoenix（注册全局 tracer）

和 `agent1_phoneix_lab.ipynb` 一样的步骤：加载仓库根目录 `.env.phoenix` → `register()` → `OpenAIInstrumentor` 打补丁。

**只有一点要提前知道（和 agent1 不同）**：`run_experiment` 运行 task 时会创建**独立的 TracerProvider**，因此实验相关的 trace（`Task: <函数名>` 根 span + 内部的 `ChatCompletion` span）会整体落在 **`Experiment-<hash>` 项目**（实验专属项目，由 Phoenix 自动分配），而**不是** `.env.phoenix` 配置的 `ai-interviewer-agent-eval`。

运行完实验后在 Phoenix UI 的项目下拉框里找 `Experiment-...` 即可。


In [ ]:

def load_phoenix_env() -> None:
    """加载仓库根目录 .env.phoenix（已存在的环境变量不覆盖）。"""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / ".env.phoenix"
        if candidate.exists():
            for line in candidate.read_text().splitlines():
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    name, _, value = line.partition("=")
                    os.environ.setdefault(name.strip(), value.strip())
            print(f"已加载配置: {candidate}")
            break


load_phoenix_env()

tracer_provider = register()  # 自动发现 PHOENIX_COLLECTOR_ENDPOINT / PHOENIX_PROJECT_NAME
OpenAIInstrumentor(tracer_provider=tracer_provider).instrument()
print(f"全局 tracing 已开启 -> {os.environ.get('PHOENIX_COLLECTOR_ENDPOINT')}, 项目: {os.environ.get('PHOENIX_PROJECT_NAME')}")



## 被测 Agent：Mini Agent（工具调用循环）

教程里的 `utils.run_agent(messages)` 是课程环境提供的（跑在电商数据上）。本地复刻版用 `agent1` 同一个 mini-agent 模式：Mock 的模拟面试统计库 + 单工具 `get_interview_stats`。

**为什么「路径长度」有教学意义**：`path_length = len(消息历史)`。一次标准的「问 → 调工具 → 工具回 → 答」是 4 条消息；如果 Agent 对某个问法**不调工具直接瞎答**，就只有 2 条消息——路径短不等于好，短是因为跳过了查数据的步骤。收敛分数度量的是**同等问法之间路径长度的一致性**。

（数据全部虚构，符合本目录的脱敏要求。）


In [ ]:

MOCK_DB = {
    "Drake": {"sessions": 12, "avg_score": 82.5, "weak_topics": ["系统设计", "行为面试"]},
}

TOOLS = [
    {
        "type": "function",
        "function": {  # chat completions 的工具声明多包一层 "function"
            "name": "get_interview_stats",
            "description": "查询指定候选人的模拟面试统计，必须调用此工具才能获得真实数据",
            "parameters": {
                "type": "object",
                "properties": {"candidate": {"type": "string", "description": "候选人代号"}},
                "required": ["candidate"],
            },
        },
    }
]


def run_agent(question: str, max_rounds: int = 4) -> list:
    """运行 mini agent，返回完整消息历史（含工具调用消息），path_length = len(消息历史)。

    返回的消息列表结构与教程 utils.run_agent 一致，便于直接算路径长度。
    """
    conversation = [{"role": "user", "content": question}]
    for _ in range(max_rounds):
        resp = client.chat.completions.create(model=MODEL, messages=conversation, tools=TOOLS)
        msg = resp.choices[0].message
        calls = msg.tool_calls or []
        if not calls:
            conversation.append({"role": "assistant", "content": msg.content})
            return conversation
        # 把模型的 assistant 消息（含 tool_calls）原样回传，再附上工具结果
        conversation.append(msg.model_dump(exclude_none=True))
        for call in calls:
            args = json.loads(call.function.arguments or "{}")
            output = json.dumps(MOCK_DB.get(args.get("candidate", ""), {}), ensure_ascii=False)
            conversation.append({"role": "tool", "tool_call_id": call.id, "content": output})
    return conversation


# 冒烟：看一眼标准路径长什么样（预期 4 条消息：user → assistant(tool_calls) → tool → assistant）
smoke = run_agent("查一下 Drake 的模拟面试统计")
for m in smoke:
    role = m["role"]
    brief = m.get("content", "")[:60] or (m.get("tool_calls") or [{}])[0].get("function", {}).get("name", "")
    print(f"  {role:<10} {brief}")
print("path_length =", len(smoke))



## 创建数据集（17 个语义等价问法）

教程用「平均每单售出数量」的 17 种问法；本地复刻换成「Drake 模拟面试统计」的等价问法集——**同一个事实问题，17 种措辞**。收敛分数要的就是：不管怎么问，路径长度应该一致。

```text
create_dataset(name=..., dataframe=..., input_keys=["question"])
```


In [ ]:

convergence_questions = [
    "查一下 Drake 的模拟面试统计",
    "Drake 的模拟面试统计是多少？",
    "我想知道 Drake 的面试统计数据",
    "请查询 Drake 的模拟面试统计",
    "查查 Drake 面试的统计数据吧",
    "Drake 参加模拟面试的统计数据有吗？",
    "给我看下 Drake 的模拟面试统计",
    "请教一下 Drake 的模拟面试统计数据",
    "麻烦查一下 Drake 的模拟面试统计",
    "Drake 的模拟面试统计结果是什么？",
    "能帮我查 Drake 的模拟面试统计吗？",
    "请问 Drake 的模拟面试统计数据是多少？",
    "查询一下候选人 Drake 的模拟面试统计",
    "看看 Drake 的模拟面试统计是什么样的",
    "我想了解 Drake 的模拟面试统计数据",
    "Drake 模拟面试的统计数据是什么？",
    "请把 Drake 的模拟面试统计调出来看看",
]

convergence_df = pd.DataFrame({"question": convergence_questions})

# 客户端统一在这里初始化（base_url 从 .env.phoenix 自动发现）
px_client = Client(base_url=os.environ.get("PHOENIX_COLLECTOR_ENDPOINT", "http://127.0.0.1:6006"))

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
dataset = px_client.datasets.create_dataset(
    name=f"convergence_questions-{now}",
    dataframe=convergence_df,
    input_keys=["question"],
)
print(f"数据集已创建: {dataset.name} ({dataset.example_count} 个样例, id={dataset.id})")



### 链接 Phoenix UI

打开下面这个链接可以查看刚上传的数据集；跑完实验后同样可以在这里对比实验表格。注意：L9 教程强调，每个课程的 notebook 连的是隔离的 Phoenix server；本地的就一个 server，直接在左侧项目栏切换即可。


In [ ]:

print(f"Phoenix UI: {os.environ.get('PHOENIX_COLLECTOR_ENDPOINT', 'http://127.0.0.1:6006')}")
print(f"数据集页面: {os.environ.get('PHOENIX_COLLECTOR_ENDPOINT', 'http://127.0.0.1:6006')}/datasets/{dataset.id}")



## 创建 Task：`run_agent_and_track_path`

教程的 task 以 `example: Example` 为参数。本地版**不需要 import Example**：task 单参数命名为 `example` 时，phoenix-client 会自动注入 `ExampleProxy`——它兼容旧式 `example.input.get("question")` 访问（`example.input` 是 dict），所以教程代码可以原样保留。

task 返回 dict：`{"path_length": ..., "messages": ...}`。`path_length` 就是 len(消息历史)——**同教程，user/system 消息都计入**。


In [ ]:

# helper：把消息历史转成可读的步骤串（教程原样）
def format_message_steps(messages) -> str:
    """
    Convert a list of message objects into a readable format that shows the steps taken.

    Args:
        messages (list): A list of message objects containing role, content, tool calls, etc.

    Returns:
        str: A readable string showing the steps taken.
    """
    steps = []
    for message in messages:
        role = message.get("role")
        if role == "user":
            steps.append(f"User: {message.get('content')}")
        elif role == "assistant":
            calls = message.get("tool_calls") or []
            if calls:
                for tool_call in calls:
                    steps.append(f"Assistant: Called tool '{tool_call['function']['name']}'")
            else:
                steps.append(f"Assistant: {message.get('content')}")
        elif role == "tool":
            steps.append(f"Tool response: {message.get('content')}")
    return "\n".join(steps)


def run_agent_and_track_path(example):
    """task：跑一次 agent 并记录路径长度。单参数命名 example -> 绑定完整样例对象。"""
    messages = [{"role": "user", "content": example.input.get("question")}]
    ret = run_agent(example.input.get("question"))
    return {"path_length": len(ret), "messages": format_message_steps(ret)}



## 运行实验

教程是 `run_experiment(dataset, task, experiment_name=..., experiment_description=...)`；本地版挂在 `px_client.experiments.run_experiment(...)` 下，签名几乎一致。

**运行要点**：

- 每个样例真实调用模型，17 个问法共用约 1~4 分钟；
- 每次 task 运行会自动产生一条以 `Task: run_agent_and_track_path` 为根的 trace，并在返回的 `task_runs[i]["trace_id"]` 里暴露 trace id——这是把「实验运行记录」和「trace 明细」关联起来的钥匙；
- 成功/失败次数、耗时都会汇总；失败的样例在 `task_runs[i]["error"]` 里。


In [ ]:

experiment = px_client.experiments.run_experiment(
    dataset=dataset,
    task=run_agent_and_track_path,
    experiment_name="Convergence Eval",
    experiment_description="Evaluating the convergence of the agent",
    timeout=180,      # 单次 task 超时（秒）：每次最多 4 轮 LLM 调用
    retries=2,
)
print("实验项目:", experiment.get("project_name"))



## 查看实验结果

教程的 `experiment.as_dataframe()` 在新版客户端里没有了（返回的是 `RanExperiment` TypedDict）。手动从 `experiment["task_runs"]` 组装一个等价的 DataFrame：

| 教程列 | 本地取值 |
|---|---|
| `example_id` | `task_run["dataset_example_id"]` |
| `output` | `task_run["output"]`（我们 task 返回的 dict） |
| `trace_id` | `task_run["trace_id"]`（TRACE_ID 教学点） |


In [ ]:

task_runs = pd.DataFrame(experiment["task_runs"])
task_runs["output"] = task_runs["output"].apply(lambda o: o if isinstance(o, dict) else {})
task_runs["path_length"] = task_runs["output"].map(lambda o: o.get("path_length"))
task_runs["steps"] = task_runs["output"].map(lambda o: o.get("messages"))

# 展示：与教程 experiment.as_dataframe() 等价的视图
result_df = task_runs[["dataset_example_id", "path_length", "steps", "trace_id"]]
print(f"共 {len(result_df)} 个样例完成，失败: {int(task_runs['error'].notna().sum())}")
result_df[["dataset_example_id", "path_length", "trace_id"]]



## 计算最优路径长度（S_optimal）

与教程一致：全批最短路径作为最优基准。注意这里的注释——**最优是「最少消息数」吗？** 是，但不代表「最优行为」。较短路径可能是「没查数据直接回答」；收敛分数只看**出一致性**，一致性差才有教学价值，一致性本身不评价正确性。


In [ ]:

# 教程写法：min over path_length（排除 None）
optimal_path_length = min(
    pl for pl in task_runs["path_length"] if pl is not None
)
print(f"The optimal path length is {optimal_path_length}")



## 收敛评估器

教程用 `@create_evaluator(name="Convergence Eval", kind="CODE")` 装饰器；本地版直接传普通函数即可（名字自动用函数名），返回 float / dict / 元组都可以。

**PPT 公式 vs 教程代码的差异（教学点）**：

- PPT 公式：$\min(1,\ S_{optimal}/S_{agent,i})$——有 clamp；
- 教程代码：`optimal_path_length / float(output.get("path_length"))`——没有 clamp。

因为 `optimal` 定义为全批最小，比值天然 ≤ 1，clamp 是防御性的（比如 future 换用「最短」以外的基准，或 S_agent 出现异常值）。本地版按 PPT 公式带 clamp，更稳妥。

**评估器返回值（教学点）**：新版解析规则是 `(score, label)` 二元组或 `(score, label, explanation)` 三元组——想带解释必须给**三元组**（`(score, explanation)` 二元组会把第二项当 label，解释丢失）。下面用 dict 写法，最直观、不歧义。


In [ ]:

def evaluate_path_length(output: dict) -> dict:
    """收敛评估器（CODE kind）：path_length 越接近最优 → 分数越接近 1。

    返回 dict {score, label, explanation}（新版也支持 (score, label, explanation) 三元组；
    注意二元组 `(score, explanation)` 会被解析为 (score, label)，explanation 会丢）。
    """
    if output and output.get("path_length"):
        score = min(1.0, optimal_path_length / float(output["path_length"]))
        return {
            "score": score,
            "label": "converged" if score >= 1.0 else "diverged",
            "explanation": f"optimal={optimal_path_length}, agent={output['path_length']}",
        }
    return {"score": 0.0, "label": "missing", "explanation": "no path_length in output"}


# 单参数 output 绑定 task 的返回 dict；函数名成为评估器名
experiment = px_client.experiments.evaluate_experiment(
    experiment=experiment,
    evaluators=[evaluate_path_length],
)
print("评估完成，evaluation_runs:", len(experiment["evaluation_runs"]))



## 汇总收敛分数

`evaluation_runs` 是 `ExperimentEvaluationRun` **dataclass 列表**（不是 dict），字段为 `experiment_run_id / name / result / error / trace_id`；`result` 是 TypedDict，含 `score`、`label`、`explanation`。

总收敛分数 = 各次运行得分的均值（等价于 PPT 公式的 Overall Convergence Score）。

另外把每个样例的 **path_length 分布**打出来：理想情况是全部相等（分数 1.0）；出现离散的值，就点开对应 trace 看是不是「没查数据直接回答」（短）或「多次工具调用才定位到正确工具」（长）。


In [ ]:

eval_rows = []
for run in experiment["evaluation_runs"]:
    r = run.result or {}
    eval_rows.append({
        "experiment_run_id": run.experiment_run_id,
        "name": run.name,
        "score": r.get("score"),
        "explanation": r.get("explanation", ""),
        "error": run.error,
        "trace_id": run.trace_id,
    })
eval_df = pd.DataFrame(eval_rows)

print(f"总收敛分数: {eval_df['score'].mean():.3f}  (n={len(eval_df)})")
print("\n逐样例得分与路径长度:")
summary = task_runs[["id", "dataset_example_id", "path_length"]].merge(
    eval_df, left_on="id", right_on="experiment_run_id", how="left"
)
summary = summary[["dataset_example_id", "path_length", "score", "explanation", "trace_id"]]
summary



## 去 Phoenix 里核对

浏览器打开 <http://127.0.0.1:6006>，**项目下拉框选择 `Experiment-<hash>`**（就是上面 `实验项目:` 打印的那个，不是 `ai-interviewer-agent-eval`），逐条检查：

1. **Trace 数量**：每次 task 运行一条根 trace（`Task: run_agent_and_track_path`），共 17 条；展开能看到嵌套的 `ChatCompletion` span（每次 LLM 调用一个）；
2. **Span Attributes**：`llm.input_messages` / `llm.output_messages`（完整 prompt 与回复）、`llm.model_name`、`llm.token_count.*`；
3. **trace_id 关联**：`task_runs[i]["trace_id"]` 与 UI 里 trace 详情页的 ID 一致——实验记录和 trace 明细是同一根线；
4. **数据集的实验对比页**：`/datasets/<id>/experiments` 能看到每个样例的 path_length / score / explanation 表格，还可与以后重跑的实验横向对比。

## 教学点小结

1. **实验三件套**：Dataset（语义等价问法）→ Task（跑一次 agent，输出路径长度）→ Evaluator（收敛评分），在本地是 `datasets.create_dataset` + `experiments.run_experiment` + `experiments.evaluate_experiment`；
2. **trace 归属**：实验运行期间的 trace 独立落进 `Experiment-<hash>` 项目（run_experiment 内建独立 TracerProvider）；全局 `register()` 的项目（`ai-interviewer-agent-eval`）只收非实验的调用——排查 trace 先看对项目；
3. **as_dataframe 消失**：新版返回 `RanExperiment` TypedDict，用 `task_runs` / `evaluation_runs` 手动组装；
4. **评估器无装饰器**：直接传函数，支持 `float` / `(score, explanation)` / dict 三种返回；
5. **收敛分数的解释边界**：分数衡量「一致」不是「正确」。分数低要追问：短路径是跳过查数，长路径是多绕了几次工具？用 trace 定位到具体样例再下结论——这正是「去 Phoenix 核对」存在的意义。
